# Non-pregnant anemia prevalence and DALYs averted by fortification

We take a "multiplication model" approach here, shifting continuous hemoglobin (as estimated
by GBD) and seeing what impact that has on anemia.

It's important to note that we directly use hemoglobin estimates, which are the first step
of the GBD anemia estimation pipeline. The risks and causes that are related to anemia are
all calculated downstream from this.

In [ ]:
import gbd_mapping
import risk_distributions
import pathlib
import pandas as pd, numpy as np
import vivarium_inputs
from vivarium_inputs import utility_data, globals as vi_globals, utilities as vi_utils
from vivarium_gbd_access import gbd
import os, contextlib, warnings, loguru
from lsff_utils.hemoglobin_distribution import hemoglobin_cdf_from_mean_sd

from vivarium_inputs.validation.raw import DataDoesNotExistError, DataAbnormalError
from tqdm.notebook import tqdm

In [ ]:
pd.set_option("display.max_columns", 30)

In [ ]:
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [ ]:
location = "nigeria"
vehicle = "bouillon"
intervention_scenario = "intervention"

## Setup and scenarios

In [ ]:
index_cols = ["sex", "age_start", "age_end", "wealth_quintile"]

age_group_ids = [
    2,3,
    388,389,
    6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235
]
sex_ids = [1,2]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [ ]:
effective_baseline_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/baseline_fortification/effective_coverage/{location}.csv')
)
assert (effective_baseline_coverage.vehicle_name == vehicle).all()
effective_baseline_coverage = effective_baseline_coverage.drop(columns=["vehicle_name"])
effective_baseline_coverage

In [ ]:
def expand(df):
    for col in sorted(list(set(df.columns) - {'value'})):
        if df[col].isnull().any():
            df = pd.concat([
                df[df[col].notnull()],
                *[df[df[col].isnull()].assign(**{col: value}) for value in df[df[col].notnull()][col].unique()]
            ])
    
    return df

In [ ]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_baseline_coverage.columns:
        effective_baseline_coverage[col] = fill_value
    else:
        effective_baseline_coverage[col] = effective_baseline_coverage[col].fillna(fill_value)

In [ ]:
effective_baseline_coverage = expand(effective_baseline_coverage)
effective_baseline_coverage

In [ ]:
effective_counterfactual_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv')
)
assert (effective_counterfactual_coverage.vehicle_name == vehicle).all()
effective_counterfactual_coverage = effective_counterfactual_coverage.drop(columns=["vehicle_name"])
effective_counterfactual_coverage

In [ ]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_counterfactual_coverage.columns:
        effective_counterfactual_coverage[col] = fill_value
    else:
        effective_counterfactual_coverage[col] = effective_counterfactual_coverage[col].fillna(fill_value)

In [ ]:
effective_counterfactual_coverage = expand(effective_counterfactual_coverage)
effective_counterfactual_coverage

In [ ]:
population = (
    pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
)

In [ ]:
non_pregnant_pop = population.pipe(lambda df: df[df.pregnant == "not_pregnant"]).drop(columns="pregnant")
non_pregnant_pop = non_pregnant_pop.set_index([c for c in non_pregnant_pop.columns if c != 'value']).value
non_pregnant_pop

In [ ]:
population.set_index(["sex", "age_start", "age_end"]).loc[("Female", 25, 30)].value.sum()

In [ ]:
non_pregnant_pop.loc[("Female", 25, 30)]['lowest']

In [ ]:
non_pregnant_pop.loc[("Female", 25, 30)].sum()

In [ ]:
population_age_groups = non_pregnant_pop.reset_index()[["age_start", "age_end"]].drop_duplicates().sort_values("age_start")
population_age_groups

In [ ]:
def map_to_population_age_groups(df):
    result = (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )
    return result

In [ ]:
effective_baseline_coverage = map_to_population_age_groups(effective_baseline_coverage).set_index([c for c in effective_baseline_coverage.columns if c != 'value']).value
effective_counterfactual_coverage = map_to_population_age_groups(effective_counterfactual_coverage).set_index([c for c in effective_counterfactual_coverage.columns if c != 'value']).value

In [ ]:
if "sex" not in effective_baseline_coverage.index.names:
    # Assume does not vary
    effective_baseline_coverage = pd.concat([
        effective_baseline_coverage.reset_index().assign(sex="Female").set_index(["wealth_quintile", "sex", "age_start", "age_end"]).value,
        effective_baseline_coverage.reset_index().assign(sex="Male").set_index(["wealth_quintile", "sex", "age_start", "age_end"]).value,
    ])

In [ ]:
effective_baseline_coverage = effective_baseline_coverage.reset_index().set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value

In [ ]:
if "sex" not in effective_counterfactual_coverage.index.names:
    # Assume does not vary
    effective_counterfactual_coverage = pd.concat([
        effective_counterfactual_coverage.reset_index().assign(sex="Female").set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value,
        effective_counterfactual_coverage.reset_index().assign(sex="Male").set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value,
    ])

In [ ]:
effective_counterfactual_coverage = effective_counterfactual_coverage.reset_index().set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value

In [ ]:
effective_counterfactual_coverage.loc[("Female", 25, 30, "lowest")]

In [ ]:
effective_baseline_coverage.loc[("Female", 25, 30, "lowest")]

In [ ]:
effective_counterfactual_coverage.loc[("Female", 25, 30, "lowest")]

In [ ]:
delta_effective_coverage = effective_counterfactual_coverage.sub(effective_baseline_coverage)
delta_effective_coverage

In [ ]:
delta_effective_coverage.loc[("Female", 25, 30, "lowest")]

In [ ]:
def reshape_to_vivarium_format(df, location):
    df = vi_utils.reshape(df, value_cols=[c for c in df.columns if 'draw_' in c])
    df = vi_utils.scrub_gbd_conventions(df, location)
    df = vi_utils.split_interval(df, interval_column="age", split_column_prefix="age")
    df = vi_utils.split_interval(df, interval_column="year", split_column_prefix="year")
    df = vi_utils.sort_hierarchical_data(df)
    df.index = df.index.droplevel("location")
    return df

## Pull GBD hemoglobin distributions

In [ ]:
me_ids = {
    "hemoglobin_mean": 10487,
    "hemoglobin_sd": 10488,
}

In [ ]:
def get_modelable_entity_draws(me_id, location):
    location_id = utility_data.get_location_id(location.title())
    result = gbd.get_modelable_entity_draws(me_id=me_id, location_id=location_id, year_id=2021)
    return (
        reshape_to_vivarium_format(result, location.title())
            .droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS]
            .copy()
    )

In [ ]:
hgb_mean = get_modelable_entity_draws(me_ids["hemoglobin_mean"], location)
hgb_mean

In [ ]:
hgb_mean.loc[("Female", 25, 30)].mean()

In [ ]:
hemoglobin_mean_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv')
hemoglobin_mean_disparities = (
    map_to_population_age_groups(hemoglobin_mean_disparities[hemoglobin_mean_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_mean_disparities

In [ ]:
wealth_quintile_probabilities = pd.read_csv(f'../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv')
wealth_quintile_probabilities

In [ ]:
wealth_quintile_probabilities = (
    map_to_population_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end"])
)
wealth_quintile_probabilities.columns.name = 'wealth_quintile'
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

In [ ]:
assert np.allclose(wealth_quintile_probabilities.groupby(["sex", "age_start", "age_end"]).sum(), 1.0)

In [ ]:
def distribute_by_disparities(df, disparities):
    pre_disparity_groups = df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum()
    print('Before distributing by disparities:')
    display(pre_disparity_groups)

    df = df.mul(disparities, axis=0)

    scale_factor = pre_disparity_groups / df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum()
    print(f'Scale factor: {scale_factor}')

    df = df * scale_factor

    assert np.allclose(
        df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum(),
        pre_disparity_groups,
    )

    return df

In [ ]:
hgb_mean = distribute_by_disparities(hgb_mean, hemoglobin_mean_disparities)

In [ ]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean")

In [ ]:
hgb_sd = get_modelable_entity_draws(me_ids["hemoglobin_sd"], location)
hgb_sd

In [ ]:
hemoglobin_sd_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/sd_disparities/{location}.csv')
hemoglobin_sd_disparities = (
    map_to_population_age_groups(hemoglobin_sd_disparities[hemoglobin_sd_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_sd_disparities

In [ ]:
hgb_sd = distribute_by_disparities(hgb_sd, hemoglobin_sd_disparities)
hgb_sd

In [ ]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd")
hgb_sd

In [ ]:
hgb_mean.loc[("Female", 25, 30, "lowest")].mean()

In [ ]:
hgb_sd.loc[("Female", 25, 30, "lowest")].mean()

## Effect size and adjustment for iron responsiveness

We assume that our overall effect size is composed of two parts:
some people respond to iron with a constant shift (no individual heterogeneity)
and other people are "not responsive" and their hemoglobin doesn't
change at all.

This is similar to how GBD models the iron deficiency risk factor.
I believe we got the lists of sequelae below from them.

As a rough approximation, we assume that our mean difference value
(from the literature) was from a population that had the global prevalence
split between iron-responsive and non-iron-responsive.

**Note: We assume everyone who is not anemic is iron-responsive.**

In [ ]:
fortification_hemoglobin_mean_difference = (
    pd.read_csv('../0100_data_prep/results/iron/fortification_hemoglobin_effects.csv')
        .set_index('vehicle_name').value
        .loc[vehicle]
)
fortification_hemoglobin_mean_difference

In [ ]:
# Cleaned this up from https://github.com/ihmeuw/vivarium_research_lsff/blob/1cb465a752d299401ae366db537dc8d557162184/multiplication_models/iron_model_U5.ipynb,
# but have not checked it in extreme detail.
iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.moderate_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.severe_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.mild_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.mild_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.moderate_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.severe_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.mild_iron_deficiency_anemia,
    gbd_mapping.sequelae.moderate_iron_deficiency_anemia,
    gbd_mapping.sequelae.severe_iron_deficiency_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.menstrual_disorders_with_mild_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_moderate_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_mild_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_moderate_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_mild_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_moderate_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_severe_anemia,
    gbd_mapping.sequelae.crohns_disease_with_mild_anemia,
    gbd_mapping.sequelae.crohns_disease_with_moderate_anemia,
    gbd_mapping.sequelae.crohns_disease_with_severe_anemia,
    gbd_mapping.sequelae.complicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.complicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_severe_anemia,
]

In [ ]:
non_iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.severe_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.mild_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.mild_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_malaria_with_mild_anemia,
    gbd_mapping.sequelae.severe_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.severe_malaria_with_severe_anemia,
    gbd_mapping.sequelae.mild_malaria_with_mild_anemia,
    gbd_mapping.sequelae.mild_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.mild_malaria_with_severe_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_mild_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_severe_anemia,
    gbd_mapping.sequelae.early_hiv_with_mild_anemia,
    gbd_mapping.sequelae.early_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.early_hiv_with_severe_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_mild_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_severe_anemia,
    gbd_mapping.sequelae.aids_with_mild_anemia,
    gbd_mapping.sequelae.aids_with_moderate_anemia,
    gbd_mapping.sequelae.aids_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_vivax_pvpr,
]

In [ ]:
len(iron_responsive_anemia_sequelae)

In [ ]:
len(non_iron_responsive_anemia_sequelae)

In [ ]:
def pull_sequelae_prevalence(location, sequelae):
    result = 0
    # There are tons of validation warnings -- look into these more?
    loguru.logger.disable("vivarium_inputs.validation.raw")
    for sequela in sequelae:
        try:
            sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location"])
        # There are even some errors, caused by all-zero values
        except DataDoesNotExistError as e:
            assert 'zero' in str(e)
            continue
        except DataAbnormalError as e:
            assert 'zero' in str(e)
            continue

        # AFAIK these are not mutually exclusive; standard GBD assumption is independence
        result += sequela_prevalence * (1 - result)
    
    loguru.logger.enable("vivarium_inputs.validation.raw")
    
    return result

In [ ]:
global_population = vivarium_inputs.get_population_structure("Global").droplevel("location").value
global_population

In [ ]:
global_non_responsive = pull_sequelae_prevalence("Global", non_iron_responsive_anemia_sequelae)
global_non_responsive

In [ ]:
global_non_responsive_aggregated = global_non_responsive.mul(global_population, axis=0).sum() / global_population.sum()
global_non_responsive_aggregated.index.name = "draw"
global_non_responsive_aggregated

In [ ]:
global_non_responsive_aggregated.describe()

In [ ]:
global_non_responsive_aggregated.mean()

In [ ]:
# mean difference observed = 0 * non-responsive + mean_difference_responsive * (1 - non-responsive)
# Assume observed in total population (some studies in the meta-analysis only included children,
# but we are applying the effect to total population anyway)
hemoglobin_effect_among_responsive = fortification_hemoglobin_mean_difference / (1 - global_non_responsive_aggregated)
hemoglobin_effect_among_responsive

In [ ]:
hemoglobin_effect_among_responsive.mean()

## Iron-responsiveness in population of interest

Note: unlike the previous, we do this *as a fraction of the anemic population*.
That is because it is only the anemic population where our shifting intervention
makes a difference (in prevalence/YLDs).

In [ ]:
iron_responsive_prevalence = pull_sequelae_prevalence(location, iron_responsive_anemia_sequelae)
iron_responsive_prevalence

In [ ]:
non_iron_responsive_prevalence = pull_sequelae_prevalence(location, non_iron_responsive_anemia_sequelae)
non_iron_responsive_prevalence

In [ ]:
iron_responsive_proportion = iron_responsive_prevalence / (iron_responsive_prevalence + non_iron_responsive_prevalence)
iron_responsive_proportion

In [ ]:
iron_responsive_proportion.columns.name = "draw"
iron_responsive_proportion = iron_responsive_proportion[DRAWS].stack()
iron_responsive_proportion

In [ ]:
assert (iron_responsive_proportion <= 1).all()

In [ ]:
iron_responsive_proportion.sort_values()

In [ ]:
iron_responsive_proportion.loc[("Female", 25, 30)].mean()

## Apply fortification effect

In [ ]:
hgb_mean_iron_responsive_with_fort = hgb_mean.add(hemoglobin_effect_among_responsive)
hgb_mean_iron_responsive_with_fort

In [ ]:
thresholds = reshape_to_vivarium_format(
    pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv'),
    location.title(),
).droplevel(["age_group_name", "grp"]).reset_index()
thresholds

In [ ]:
assert (
    (thresholds.hgb_upper_mild == thresholds.hgb_upper_anemic).all() &
    (thresholds.hgb_lower_severe == thresholds.hgb_lower_anemic).all()
)
thresholds = thresholds.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [ ]:
assert (
    (thresholds.hgb_lower_mild == thresholds.hgb_upper_moderate).all() &
    (thresholds.hgb_lower_moderate == thresholds.hgb_upper_severe).all()
)
thresholds = thresholds.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [ ]:
thresholds = thresholds.set_index(["sex", "age_start", "age_end", "pregnant"])
thresholds

In [ ]:
def calculate_anemia_from_mean_sd_hemoglobin(mean, sd):
    orig_index = mean.index
    result = mean.reset_index().merge(sd.reset_index(), how="outer", validate="m:1").assign(pregnant=0).merge(thresholds.reset_index(), how="left", validate="m:1")

    cdf = hemoglobin_cdf_from_mean_sd(result["mean"], result.sd)

    result["severe"] = cdf(result.hgb_upper_severe.copy()) - cdf(result.hgb_lower_severe.copy())
    result["moderate"] = cdf(result.hgb_upper_moderate.copy()) - result["severe"].copy()
    result["mild"] = cdf(result.hgb_upper_mild.copy()) - result["moderate"].copy() - result["severe"].copy()
    result["anemic"] = result["mild"] + result["moderate"] + result["severe"]
    
    return result.set_index(orig_index.names)[["severe", "moderate", "mild", "anemic"]]

In [ ]:
baseline_anemia = calculate_anemia_from_mean_sd_hemoglobin(hgb_mean, hgb_sd)
baseline_anemia

In [ ]:
iron_responsive_with_fort_anemia = calculate_anemia_from_mean_sd_hemoglobin(hgb_mean_iron_responsive_with_fort.rename("mean"), hgb_sd)
iron_responsive_with_fort_anemia

In [ ]:
(baseline_anemia.loc[("Female", 25, 30, "lowest")].mean() - iron_responsive_with_fort_anemia.loc[("Female", 25, 30, "lowest")].mean()) / baseline_anemia.loc[("Female", 25, 30, "lowest")].mean()

In [ ]:
affected_by_intervention = (delta_effective_coverage * iron_responsive_proportion).droplevel(["year_start", "year_end"])
affected_by_intervention

In [ ]:
affected_by_intervention.loc[("Female", 25, 30, "lowest")].mean()

In [ ]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part that received new effective coverage) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part, since they all depend on CDFs which combine
# this way.

intervention_anemia = baseline_anemia.mul(1 - affected_by_intervention, axis=0) + iron_responsive_with_fort_anemia.mul(affected_by_intervention, axis=0) 
intervention_anemia

In [ ]:
from lsff_utils.hemoglobin_distribution import hemoglobin_pdf_from_mean_sd


baseline_pdf = hemoglobin_pdf_from_mean_sd(np.array([hgb_mean.loc[("Female", 25, 30, "lowest")].mean()]), np.array([hgb_sd.loc[("Female", 25, 30, "lowest")].mean()]))
baseline_pdf

In [ ]:
benefit_pdf = hemoglobin_pdf_from_mean_sd(np.array([hgb_mean_iron_responsive_with_fort.loc[("Female", 25, 30, "lowest")].mean()]), np.array([hgb_sd.loc[("Female", 25, 30, "lowest")].mean()]))
benefit_pdf

In [ ]:
def intervention_pdf(x):
    proportion_that_benefit = affected_by_intervention.loc[("Female", 25, 30, "lowest")].mean()
    return (
        baseline_pdf(x) * (1 - proportion_that_benefit) +
        benefit_pdf(x) * proportion_that_benefit
    )

In [ ]:
import matplotlib.pyplot as plt

x_values = np.linspace(60, 160, 100)
with np.errstate(under="ignore"):
    plt.plot(x_values, [baseline_pdf(x) for x in x_values], label="Baseline hemoglobin")
    plt.plot(x_values, [benefit_pdf(x) for x in x_values], label="Hemoglobin among those who benefit")
    plt.plot(x_values, [intervention_pdf(x) for x in x_values], label="Hemoglobin in intervention scenario")
    plt.vlines(thresholds.loc[("Female", 25, 30, 0)].hgb_upper_mild, 0, 0.03, linestyles="dashed", label="Anemia", color="lime")
    plt.vlines(thresholds.loc[("Female", 25, 30, 0)].hgb_upper_moderate, 0, 0.03, linestyles="dashed", label="Moderate anemia", color="gold")
    plt.vlines(thresholds.loc[("Female", 25, 30, 0)].hgb_upper_severe, 0, 0.03, linestyles="dashed", label="Severe anemia", color="crimson")
    plt.xlabel("Hemoglobin (g/L)")
    plt.ylabel("Probability density")
    plt.title(f"Hemoglobin before and after {vehicle} fortification intervention in non-pregnant females 25-30 years in the bottom quintile, {location.title()}")
    plt.legend(bbox_to_anchor=(1.05, 1))

In [ ]:
(baseline_anemia - intervention_anemia).sort_values("anemic")

In [ ]:
def anemia_to_yld_rates(anemia):
    disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
    disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
    disability_weights.columns.name = 'draw'
    disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
    display(disability_weights)

    orig_index = anemia.index
    anemia = anemia.reset_index().merge(
        disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
        validate="m:1",
    ).merge(
        disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
        validate="m:1",
    ).merge(
        disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
        validate="m:1",
    )

    anemia["mild_yld_rate"] = anemia.mild * anemia.mild_dw
    anemia["moderate_yld_rate"] = anemia.moderate * anemia.moderate_dw
    anemia["severe_yld_rate"] = anemia.severe * anemia.severe_dw
    anemia['anemic_yld_rate'] = anemia['mild_yld_rate'] + anemia['moderate_yld_rate'] + anemia['severe_yld_rate']

    return anemia.set_index(orig_index.names).filter(like='yld_rate')

In [ ]:
baseline_anemia_yld_rates = anemia_to_yld_rates(baseline_anemia)
baseline_anemia_yld_rates

In [ ]:
(baseline_anemia_yld_rates.loc[("Female", 25, 30, "lowest")].mean() - anemia_to_yld_rates(iron_responsive_with_fort_anemia).loc[("Female", 25, 30, "lowest")].mean()) / baseline_anemia_yld_rates.loc[("Female", 25, 30, "lowest")].mean()

In [ ]:
intervention_anemia_yld_rates = anemia_to_yld_rates(intervention_anemia)
intervention_anemia_yld_rates

In [ ]:
(baseline_anemia_yld_rates - intervention_anemia_yld_rates).sort_values("anemic_yld_rate")

In [ ]:
assert ((baseline_anemia_yld_rates - intervention_anemia_yld_rates).anemic_yld_rate > 0).all()

In [ ]:
baseline_ylds = (baseline_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1) * non_pregnant_pop)
baseline_ylds

In [ ]:
intervention_ylds = (intervention_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1) * non_pregnant_pop)
intervention_ylds

In [ ]:
(baseline_ylds.loc[("Female", 25, 30, "lowest")].mean() - intervention_ylds.loc[("Female", 25, 30, "lowest")].mean())

In [ ]:
baseline_ylds.groupby(["wealth_quintile"]).sum() - intervention_ylds.groupby(["wealth_quintile"]).sum()

In [ ]:
ylds = pd.concat([
    baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
    intervention_ylds.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

In [ ]:
results_dir = f'./results/{vehicle.lower()}/{location.lower()}/{intervention_scenario.lower()}'

In [ ]:
path = f'{results_dir}/ylds.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylds.to_parquet(path)

In [ ]:
baseline_anemia_prevalence = baseline_anemia['anemic'].unstack("draw").mean(axis=1)
baseline_anemia_prevalence

In [ ]:
baseline_anemia_cases = baseline_anemia_prevalence.mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

In [ ]:
baseline_anemia_cases.sum() / non_pregnant_pop.sum()

In [ ]:
baseline_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

In [ ]:
intervention_anemia_prevalence = intervention_anemia['anemic'].unstack("draw").mean(axis=1)
intervention_anemia_prevalence

In [ ]:
anemia_prevalence = pd.concat([
    baseline_anemia_prevalence.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_prevalence.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_prevalence

In [ ]:
path = f'{results_dir}/anemia_prevalence.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_prevalence.to_parquet(path)

In [ ]:
intervention_anemia_cases = intervention_anemia_prevalence.mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

In [ ]:
intervention_anemia_cases.sum() / non_pregnant_pop.sum()

In [ ]:
intervention_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

In [ ]:
(baseline_anemia_cases.groupby(["wealth_quintile"]).sum() - intervention_anemia_cases.groupby(["wealth_quintile"]).sum()).map(lambda x: f'{round(x):,.0f}')

In [ ]:
anemia_cases = pd.concat([
    baseline_anemia_cases.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_cases.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_cases

In [ ]:
path = f'{results_dir}/anemia_cases.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_cases.to_parquet(path)